<a href="https://colab.research.google.com/github/kianushmaleki/Beta_Bank/blob/main/Extract_Key_Dates_from_SDF_Documents_1_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Resources

# PDF Data Extraction: Dates, Graphics, and Tables
This notebook provides a comprehensive pipeline for extracting structured data from PDF documents, specifically targeting Dates, Tables, and Graphical Elements (Vector and Raster).

**Key Features:**
*   **Intelligent Date Classification:** Categorizes dates based on context (e.g., Expiration vs. Manufacturing).
*   **Advanced Table Extraction:** Uses vertical gap analysis and horizontal alignment to group text into tables.
*   **Visual Auditing:** Generates highlighted PDFs to verify extracted entities.
*   **Structured Export:** Compiles all findings into a master JSON file.

## 1. Setup and Configuration
In this section, we install necessary dependencies, import all required modules once, and compile our regular expression patterns for high-performance searching.

In [ ]:
# Install the required library for PDF processing
!pip install pymupdf pandas -q

import os
import json
import re
import fitz
import pandas as pd
from collections import Counter
from google.colab import files

# --- Global Regex Patterns ---
# Identifies various date formats including DD-MMM-YYYY and YYYYMMDD
MASTER_PATTERN = re.compile(
    r"\b(?:\d{1,2}[-\s](?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*[-\s,]*\d{4}|20\d{6})\b",
    re.I,
)
LETTER_PATTERN = re.compile(r"\b(?:dear|best regards|sincerely)\b", re.I)
COQ_PATTERN = re.compile(r"\bcertificate of quality\b", re.I)
PRODUCT_PATTERN = re.compile(r"Product(?::|\s{2,})\s*(.+)", re.I)

print("Environment and Patterns Ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 29.6 MB/s eta 0:00:00
Environment and Patterns Ready.


## 2. Core Extraction Functions
These functions handle the heavy lifting: classifying date types, locating dates with precise coordinates, and identifying multi-column table structures.

In [ ]:
# Centralized configuration for easy tuning
CONFIG = {
    "GAP_THRESHOLD": 20,    # Vertical distance to separate tables
    "TOLERANCE": 1.5,       # Vertical snapping for text lines
    "MIN_ROWS": 2,          # Filters out single-line noise/footers
    "X_TOLERANCE": 3.0      # Fuzzy horizontal column alignment (in points)
}

In [ ]:
def find_vendor_info(page: fitz.Page) -> dict:
    """
    Extracts vendor and signatory details specifically from 'letter' type pages.
    """
    text = page.get_text("text")
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    # Initialize default structure
    vendor_data = {
        "vendor_info": {"company": "Unknown", "address": "Unknown", "website": "Unknown"},
        "signatory": {"name": "Unknown", "email": "Unknown"}
    }

    if not lines:
        return vendor_data

    # 1. Company and Address (First 4 lines)
    vendor_data["vendor_info"]["company"] = lines[0]
    if len(lines) >= 4:
        vendor_data["vendor_info"]["address"] = ", ".join(lines[1:4])

    # 2. Website Detection
    web_match = re.search(r"www\.[a-z0-9\.]+|[a-z0-9\.]+\.com", text, re.I)
    if web_match:
        vendor_data["vendor_info"]["website"] = web_match.group()

    # 3. Signatory Logic (Email and preceding lines)
    email_match = re.search(r"[\w\.-]+@[\w\.-]+", text)
    if email_match:
        vendor_data["signatory"]["email"] = email_match.group()
        # Find the line containing the email to look backwards
        for idx, line in enumerate(lines):
            if email_match.group() in line:
                if idx >= 2:
                    # Capture the two lines before the email
                    vendor_data["signatory"]["name"] = f"{lines[idx-2]} {lines[idx-1]}"
                break

    return vendor_data

In [ ]:
def classify_date_type(text_before: str, text_after: str) -> str:
    """Classifies the type of date based on surrounding text."""
    lower_before = text_before.lower()
    lower_after = text_after.lower()

    if "effective date" in lower_before:
        return "Effective Date"
    elif any(x in lower_before for x in ["manufacturing date", "date of manufacture"]):
        return "Manufacturing Date"
    elif "revision date" in lower_before:
        return "Revision Date"
    elif any(x in lower_before or x in lower_after for x in ["docusign", "signed"]):
        return "DocuSign Timestamp"
    elif "expiration date" in lower_before:
        return "Expiration Date"
    return "General Date"

def extract_all_dates(page: fitz.Page) -> list[dict]:
    """Extracts dates and their surrounding context from a PDF page."""
    output = []
    current_page_num = page.number + 1
    text = page.get_text("text")

    # Detect page type
    if LETTER_PATTERN.search(text):
        page_type = "letter"
        product_info = ""
    elif COQ_PATTERN.search(text):
        page_type = "CoQ"
        product_matches = PRODUCT_PATTERN.findall(text)
        product_info = product_matches[0].strip() if product_matches else "Not Found"
    else:
        page_type = "Unknown"
        product_info = ""

    for match in MASTER_PATTERN.finditer(text):
        date_found = match.group().strip()
        start_pos, end_pos = match.start(), match.end()

        # Get precise coordinates
        precise_matches = page.search_for(date_found)
        if precise_matches:
            final_rect = precise_matches[0]
            for rect in precise_matches[1:]:
                if abs(rect.y0 - final_rect.y0) < 5:
                    final_rect |= rect
            bbox = [round(final_rect.x0, 2), round(final_rect.y0, 2),
                    round(final_rect.x1, 2), round(final_rect.y1, 2)]
        else:
            bbox = [0, 0, 0, 0]

        all_lines_before = text[:start_pos].splitlines()
        before_text = all_lines_before[-1].strip() if all_lines_before else ""
        after_text = text[end_pos:min(end_pos + 50, len(text))]

        output.append({
            "page_type": page_type,
            "related product": product_info,
            "extracted_data": date_found,
            "page_number": current_page_num,
            "coordinate_box": bbox,
            "date_type": classify_date_type(before_text, after_text)
        })
    return output

In [ ]:
def extract_tables_from_document(pdf_path: str) -> dict:
    """
    Extracts tables using fuzzy horizontal alignment and modular gap analysis.
    """
    doc = fitz.open(pdf_path)
    all_page_tables_dict = {}

    for i, page in enumerate(doc):
        page_dict = page.get_text("dict")
        candidate_rows = []
        rows_by_y = {}

        # --- STEP 1: Snapping spans into lines ---
        for b in page_dict["blocks"]:
            if b['type'] == 0:
                for line in b['lines']:
                    line_text = "".join([span["text"] for span in line["spans"]]).strip()
                    y0 = line['bbox'][1]

                    matched = False
                    for existing_y in rows_by_y.keys():
                        if abs(y0 - existing_y) < CONFIG["TOLERANCE"]: # Use CONFIG
                            rows_by_y[existing_y].append({"text": line_text, "bbox": line['bbox']})
                            matched = True
                            break
                    if not matched:
                        rows_by_y[y0] = [{"text": line_text, "bbox": line['bbox']}]

        # --- STEP 2: Filter for multi-column candidates ---
        for y_coord in rows_by_y:
            if len(rows_by_y[y_coord]) > 1:
                sorted_row = sorted(rows_by_y[y_coord], key=lambda x: x["bbox"][0])
                candidate_rows.append(sorted_row)

        # --- STEP 3: Fuzzy Horizontal Alignment (Item 1) ---
        # Instead of rounding, we check proximity across all text on the page
        final_page_candidates = []
        all_x0s = [item["bbox"][0] for row in candidate_rows for item in row]

        for row in candidate_rows:
            is_valid_row = False
            for item in row:
                curr_x0 = item["bbox"][0]
                # Find any other x0 positions within the fuzzy tolerance
                matches = [x for x in all_x0s if abs(x - curr_x0) <= CONFIG["X_TOLERANCE"]]

                if len(matches) > 1: # Found itself plus at least one other aligned row
                    is_valid_row = True
                    break

            if is_valid_row:
                final_page_candidates.append(row)

        # --- STEP 4: Segment into Tables (Item 4) ---
        page_tables = []
        if final_page_candidates:
            final_page_candidates.sort(key=lambda row: row[0]["bbox"][1])
            current_table = [final_page_candidates[0]]

            for j in range(1, len(final_page_candidates)):
                # Calculate vertical gap between rows
                prev_bottom = current_table[-1][0]["bbox"][3]
                curr_top = final_page_candidates[j][0]["bbox"][1]

                if (curr_top - prev_bottom) > CONFIG["GAP_THRESHOLD"]: # Use CONFIG
                    if len(current_table) >= CONFIG["MIN_ROWS"]:
                        page_tables.append(current_table)
                    current_table = [final_page_candidates[j]]
                else:
                    current_table.append(final_page_candidates[j])

            if len(current_table) >= CONFIG["MIN_ROWS"]:
                page_tables.append(current_table)

        if page_tables:
            all_page_tables_dict[i + 1] = page_tables

    doc.close()
    return all_page_tables_dict

## 3. Visualization and Export Logic
These functions generate the final deliverables, including highlighted PDFs for visual verification and the comprehensive JSON file.

In [ ]:
def highlight_visuals(source_path: str, date_data: list, table_data: dict):
    """Generates the primary verification PDF with specific color-coded highlights."""
    doc = fitz.open(source_path)
    line_threshold = 3.0
    for i, page in enumerate(doc):
        shape = page.new_shape()
        p_num = i + 1

        # 1. Highlight Images and Drawings (RED)
        drawings = page.get_drawings()
        for d in drawings:
            rect = d["rect"]
            if rect.width >= line_threshold and rect.height >= line_threshold:
                shape.draw_rect(rect)
                shape.finish(color=(1, 0, 0), width=1)
        for img in page.get_images(full=True):
            for r in page.get_image_rects(img[0]):
                shape.draw_rect(r)
                shape.finish(color=(1, 0, 0), width=1)

        # 2. Highlight Dates (GREEN)
        page_dates = [d for d in date_data if d["page_number"] == p_num]
        for entry in page_dates:
            if entry["coordinate_box"] != [0, 0, 0, 0]:
                shape.draw_rect(entry["coordinate_box"])
                shape.finish(color=(0, 1, 0), width=1.5)

        # 3. Highlight Tables (BLUE)
        tables = table_data.get(p_num, [])
        for t in tables:
            x0s = [c["bbox"][0] for r in t for c in r]
            y0s = [c["bbox"][1] for r in t for c in r]
            x1s = [c["bbox"][2] for r in t for c in r]
            y1s = [c["bbox"][3] for r in t for c in r]
            shape.draw_rect([min(x0s), min(y0s), max(x1s), max(y1s)])
            shape.finish(color=(0, 0, 1), width=2)
        shape.commit()
    doc.save("verification_highlights.pdf")
    doc.close()

In [ ]:
def export_json(source_doc, date_data, table_data):
    """Compiles results into a structured JSON export."""
    master = []
    for i, page in enumerate(source_doc):
        p_num = i + 1
        text = page.get_text("text") # Necessary to determine page_type

        # Determine page_type to avoid NameError
        if LETTER_PATTERN.search(text):
            page_type = "letter"
        elif COQ_PATTERN.search(text):
            page_type = "CoQ"
        else:
            page_type = "Unknown"

        # Initialize the context dictionary
        if page_type == "letter":
            context_data = find_vendor_info(page)
        else:
            context_data = {}

        master.append({
            "page_number": p_num,
            "page_type": page_type, # Added for clarity
            "context": context_data, # Integrated vendor info
            "entities": [{"label": d["date_type"], "text": d["extracted_data"], "bbox": d["coordinate_box"]}
                         for d in date_data if d["page_number"] == p_num],
            "tables": [[[{"text": c["text"], "bbox": c["bbox"]} for c in r] for r in t]
                       for t in table_data.get(p_num, [])],
            "graphics": {"vector": len(page.get_drawings()), "raster": len(page.get_images())}
        })
    with open("extracted_data.json", "w") as f:
        json.dump(master, f, indent=2)

## 4. Execution and Results Summary
Run the cell below to upload your file and process the data. A summary table will display the findings per page.

In [ ]:
# 1. Upload File
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 2. Process Data
src_doc = fitz.open(filename)
all_tables = extract_tables_from_document(filename)
all_dates = []
summary = []

print(f"Processing {filename}...")
for i, page in enumerate(src_doc):
    page_num = i + 1
    text = page.get_text("text") # Get text for classification

    # --- ADDED: Vendor Info Printing Logic ---
    # Check if the current page is a letter to print vendor details
    if LETTER_PATTERN.search(text):
        vendor_details = find_vendor_info(page)
        print(f"\n--- Vendor Details Found on Page {page_num} ---")
        print(f"Company: {vendor_details['vendor_info']['company']}")
        print(f"Address: {vendor_details['vendor_info']['address']}")
        print(f"Email:   {vendor_details['signatory']['email']}")
        print(f"Contact: {vendor_details['signatory']['name']}")
        print("-" * 40)

    # Extract dates for current page
    page_dates = extract_all_dates(page)
    all_dates.extend(page_dates)

    # Log summary data for the final display
    summary.append({
        "Page": page_num,
        "Dates Found": len(page_dates),
        "Tables Found": len(all_tables.get(page_num, [])),
        "Vector Drawings": len(page.get_drawings()),
        "Pixel Images": len(page.get_images())
    })

# 3. Export Results
highlight_visuals(filename, all_dates, all_tables)
export_json(src_doc, all_dates, all_tables)
src_doc.close()

# 4. Display Results
print("\n--- EXTRACTION SUMMARY ---")
display(pd.DataFrame(summary))

print("\n--- VERIFICATION GUIDE (verification_highlights.pdf) ---")
print("* RED: Graphics (Vector drawings or Pixel images).")
print("* GREEN: Extracted Dates (Effective, Manufacturing, etc.).")
print("* BLUE: Extracted Tables.")

print("\nIMPORTANT: Please manually inspect the RED boxes. These areas contain non-text "
      "graphical data that may include critical dates or signatures.")

print("\nTask complete. Download 'extracted_data.json' and 'verification_highlights.pdf'.")

Saving sample-sdf-document-1.pdf to sample-sdf-document-1.pdf
Processing sample-sdf-document-1.pdf...

--- Vendor Details Found on Page 1 ---
Company: Cytiva
Address: 100 Results Way, Marlborough, MA 01752, United States
Email:   michaeltoner@cytiva.com
Contact: Mike Toner Product Manager
----------------------------------------

--- EXTRACTION SUMMARY ---


,Page,Dates Found,Tables Found,Vector Drawings,Pixel Images
0,1,1,1,170,1
1,2,2,1,6,1
2,3,2,2,23,0



--- VERIFICATION GUIDE (verification_highlights.pdf) ---
* RED: Graphics (Vector drawings or Pixel images).
* GREEN: Extracted Dates (Effective, Manufacturing, etc.).
* BLUE: Extracted Tables.

IMPORTANT: Please manually inspect the RED boxes. These areas contain non-text graphical data that may include critical dates or signatures.

Task complete. Download 'extracted_data.json' and 'verification_highlights.pdf'.


This section provides a summary of the strengths and weaknesses of the current PDF data extraction pipeline, along with suggestions for future enhancements and a description of the methodology used.

### Methodology: Geometric Table Extraction

The core of this pipeline is a geometric coordinate analysis method. Instead of treating the PDF as a simple stream of characters, the algorithm reconstructs the document's structure by "snapping" text spans into logical rows based on their vertical alignment. By implementing coordinate-based proximity thresholds and vertical gap thresholds, the system successfully groups misaligned columns and segments distinct data blocks, ensuring high data integrity even for complex, borderless industry tables.

### Strengths

*   **Geometric Precision**: The pipeline excels in reconstructing tables from borderless PDFs by utilizing coordinate-based "snapping." This method offers significantly higher accuracy compared to traditional text-stream extraction approaches.
*   **Flexible Alignment (Tolerance)**: The script is designed to be "forgiving" regarding object placement. By using `X_TOLERANCE` and `TOLERANCE` settings, it can correctly group columns and rows even if they are slightly misaligned (shifted by a few pixels), which prevents data from being missed due to minor PDF formatting variations.
*   **Context-Aware Labeling**: By analyzing the surrounding text (both "before" and "after" a date), the model assigns semantic meaning (e.g., distinguishing between a "Manufacturing Date" and an "Expiration Date"). This provides richer, more actionable data than merely extracting raw date strings.
*   **Configurability**: The use of a central `CONFIG` dictionary adheres to MLOps best practices, enabling easy hyperparameter tuning and adjustments without requiring modifications to the core logic of the extraction functions.

### Weaknesses

*   **Heuristic Dependency**: The rules for extracting vendor information (e.g., "first four lines") and signatory details (e.g., "two lines before email") are specific to the current document templates. These heuristics may become fragile and prone to failure if future documents deviate significantly in their header or signature layouts.
*   **Non-Searchable Content**: The pipeline relies on the presence of a text layer within the PDF. It cannot extract information from text embedded within raster images or complex vector drawings, which limits its comprehensiveness for certain document types.
*   **Coordinate Sensitivity**: While fuzzy logic mitigates some issues, extremely complex layouts or documents with overlapping tables might still present challenges that require more sophisticated segmentation algorithms.

### Future Improvements & Scaling

*   **OCR Integration (Critical)**: For truly complete data extraction, the next phase must include an Optical Character Recognition (OCR) engine (such as Tesseract or Azure Form Recognizer). This is necessary to analyze the content within the RED BOXES identified in the verification PDF, thereby capturing dates, signatures, or other critical information currently invisible to the text extractor.
*   **Regular Expression Expansion**: To accommodate diverse international documents, the `MASTER_PATTERN` for date extraction should be expanded to include a wider array of global date formats.
*   **Validation Layer**: Implementing a Pydantic schema for the extracted JSON output would ensure data integrity. This validation step would verify data types (e.g., confirming an "Expiration Date" is always a valid ISO-8601 string) before the data is ingested into a database or downstream system.

**Note**: Always meticulously review the `verification_highlights.pdf`. The presence of red boxes is a crucial indicator of areas where human oversight or OCR-assisted extraction is necessary to achieve 100% data coverage.